# DiaNutri: Data Cleaning and KNN Recommendation

This notebook is the data-science part of the project. We will inspect the supplied CSV, clean it, engineer the fields needed by the five assistant questions, save a cleaned dataset, and test the hybrid filter + KNN recommender.

**Project rule:** area, the interval from the dataset's minimum price to the user's maximum budget, sugar claim, and allergies are hard filters. KNN is used only to rank the remaining meals by preference.

## 1. Import the libraries

- `pandas` loads and analyzes tabular data.
- `matplotlib` and `seaborn` create simple charts.
- `recommender.py` contains the reusable cleaning and model functions used by both this notebook and the web page.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from recommender import clean_dataset, recommend_restaurants

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data" / "lebanon_diabetic_restaurants.csv").exists():
    PROJECT_DIR = PROJECT_DIR.parent

RAW_PATH = PROJECT_DIR / "data" / "lebanon_diabetic_restaurants.csv"
CLEAN_PATH = PROJECT_DIR / "data" / "cleaned_restaurants.csv"
print("Project directory:", PROJECT_DIR)
print("Raw dataset:", RAW_PATH)

## 2. Load and inspect the raw dataset

Never begin by changing the data. First inspect its size, headers, samples, data types, missing values, duplicates, and category spelling.

In [ ]:
raw_df = pd.read_csv(RAW_PATH)
print(f"Rows: {raw_df.shape[0]} | Columns: {raw_df.shape[1]}")
print(f"Dataset meal-price interval: ${raw_df['Min Price (USD)'].min():g}–${raw_df['Max Price (USD)'].max():g}")
display(raw_df.head())

In [ ]:
audit_table = pd.DataFrame({
    "data_type": raw_df.dtypes.astype(str),
    "missing_values": raw_df.isna().sum(),
    "unique_values": raw_df.nunique(dropna=False),
})
display(audit_table)
print("Exact duplicate rows:", int(raw_df.duplicated().sum()))
print("Rows with min price > max price:", int((raw_df["Min Price (USD)"] > raw_df["Max Price (USD)"]).sum()))
print("Rows with min calories > max calories:", int((raw_df["Min Calories"] > raw_df["Max Calories"]).sum()))

### Inspect category spelling carefully

`value_counts()` can visually hide trailing spaces. `repr()` exposes them, so `Beirut` and `Beirut ` are visibly different.

In [ ]:
print("Raw area values:")
for area, count in raw_df["Area"].value_counts(dropna=False).items():
    print(repr(area), "->", count)

## 3. Clean and enrich the dataset

`clean_dataset()` performs the full repeatable pipeline:

1. convert headers to snake case;
2. verify required columns;
3. strip and normalize text;
4. convert prices/calories to numbers;
5. remove unusable, duplicate, or impossible rows;
6. validate website formats;
7. derive meal categories, food tags, sugar-claim status, and allergen flags;
8. save the result as a new CSV without overwriting the raw data.

In [ ]:
clean_df = clean_dataset(RAW_PATH, CLEAN_PATH)
print(f"Clean rows: {len(clean_df)}")
print(f"Clean columns: {len(clean_df.columns)}")
print(f"Saved to: {CLEAN_PATH}")
display(clean_df.head())

## 4. Verify the cleaning result

A cleaning pipeline is incomplete until its output is checked. The assertions below intentionally stop the notebook if a critical rule is broken.

In [ ]:
assert clean_df["restaurant_name"].notna().all()
assert clean_df["meal_name"].notna().all()
assert (clean_df["min_price_usd"] <= clean_df["max_price_usd"]).all()
assert "Beirut " not in set(clean_df["area"])
assert clean_df.duplicated(["restaurant_name", "location", "meal_name"]).sum() == 0
print("All critical cleaning checks passed.")
print("Clean area values:", sorted(clean_df["area"].unique()))

## 5. Examine the engineered fields

The raw CSV does not contain the exact categories needed by the interface. These new columns connect the data to the questions asked by the assistant.

In [ ]:
engineered_columns = [
    "restaurant_name", "meal_name", "meal_category",
    "food_tags", "sugar_status", "sugar_eligible",
    "contains_gluten", "contains_dairy",
    "contains_tree_nuts", "contains_sesame",
]
display(clean_df[engineered_columns].head(12))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

clean_df["area"].value_counts().sort_values().plot.barh(ax=axes[0], color="#3a7d5d")
axes[0].set_title("Meals per area")
axes[0].set_xlabel("Number of rows")

clean_df["meal_category"].value_counts().plot.bar(ax=axes[1], color="#d8895b")
axes[1].set_title("Derived meal categories")
axes[1].set_xlabel("Meal category")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

## 6. Audit the sugar requirement

The dataset has no measured `total_sugar_g` column. Therefore the code extracts only written claims. It does **not** convert the word diabetic-friendly into a zero-sugar guarantee.

In [ ]:
display(clean_df["sugar_status"].value_counts().rename_axis("status").to_frame("meals"))
sugar_rows = clean_df.loc[
    clean_df["sugar_eligible"],
    ["restaurant_name", "meal_name", "sugar_status", "diabetic_notes"],
]
print("Rows eligible for the app's sugar filter:", len(sugar_rows))
display(sugar_rows)

## 7. Run an example recommendation

The function first filters the rows, including the hard price interval from the dataset minimum through `max_budget`. It never increases or ignores the user's maximum budget. It then vectorizes the remaining meal/request features and uses cosine-distance KNN. Exact food/meal matches are shown when they exist; alternatives are used only when the strict preferences have no match.

In [ ]:
example = recommend_restaurants(
    clean_df,
    area="Beirut",
    meal_choice="Lunch / Dinner",
    food_choice="Chicken / Turkey",
    max_budget=15,
    allergies=[],
    top_restaurants=4,
    meals_per_restaurant=2,
)
print(example["message"])
print("Filter audit:", example["audit"])
print("Exact preference meals:", example["exact_preference_meals"])

In [ ]:
for restaurant in example["restaurants"]:
    print("\nRestaurant name:", restaurant["restaurant_name"])
    print("Location:", restaurant["location"])
    print("Web:", restaurant["website"] or "Not available")
    for number, meal in enumerate(restaurant["meals"], start=1):
        print(f"Meal {number}: {meal['meal_name']}")
        print("Meal description:", meal["meal_description"])
        print("Price:", meal["price"])
        print("Sugar evidence:", meal["sugar_status"])

## 8. Test an allergy case

The next query excludes tree nuts before KNN. This is the correct order because allergy safety must not depend on similarity distance.

In [ ]:
allergy_example = recommend_restaurants(
    clean_df,
    area="Beirut",
    meal_choice="Any meal",
    food_choice="Any food",
    max_budget=35,
    allergies=["Tree nuts"],
)
print(allergy_example["audit"])
for restaurant in allergy_example["restaurants"]:
    print(restaurant["restaurant_name"], "->", [meal["meal_name"] for meal in restaurant["meals"]])

## 9. Final conclusions

- The current 152-row dataset is appropriate for a classroom prototype, not for medical deployment.
- KNN is useful as a transparent ranking method after strict filters.
- A larger future dataset should add verified sugar grams, carbohydrate grams, serving size, full allergen declarations, a source URL, and a last-verified date.
- Run `python -m streamlit run app.py` in the VS Code terminal to open the assistant page.
- Run `python -m unittest discover -s tests -v` to verify the project rules.